In [2]:
%pip install -q langchain langchain-community langchain-text-splitters langchain-google-genai langchain-chroma chromadb pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 

**Step - 1 : Load the documents**

In [4]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    "/content/GenAI",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

print("Documents loaded:", len(documents))

Documents loaded: 28


**Step 2: Split documents into chunks**

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Chunks created:", len(chunks))

Chunks created: 90


**Step 3: Create Embeddings**

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Gemini Api Key
f=open(r"C:\Users\siva9\OneDrive\Desktop\GenAI_Project\Google_gemini_API_key.txt")
GEMINI_API_KEY = f.read()

# Create Embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

**Step 4: Create the CHROMA Vector database**

In [7]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="bankprep_knowledge",
    persist_directory="/content/bankprep_chroma"
)

print("Documents stored in ChromaDB.")

Documents stored in ChromaDB.


**Step - 5 : Create aa Retriever Object**

In [8]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

**Step 6 : Initialize a Chat Prompt Template**

In [9]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT_TEMPLATE = """
You are BankPrep AI, a banking exam preparation assistant.

Answer the question using only the supplied context.

Context:
{context}

Question:
{question}

Instructions:
- Explain the answer clearly in simple language.
- Use step-by-step explanations for calculations when relevant.
- Do not invent facts or information that is not supported by the context.
- If the answer is not available in the context, say:
  "I could not find this information in the provided study materials."
"""

prompt_template = ChatPromptTemplate.from_template(
    PROMPT_TEMPLATE
)

**Step - 7 : Initialize Gemini chat model and output parser**

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

chat_model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.2
)

parser = StrOutputParser()

**Step 8 : Define a RAG Chain**

In [11]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | chat_model
    | parser
)

**Invoke the Chain**

In [12]:
query = "What is the current repo rate?"

answer = rag_chain.invoke(query)

print(answer)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided study materials, the current policy repo rate is **5.50 per cent** (as of 9 October 2026). 

It was increased by 25 basis points from 5.25 per cent to 5.50 per cent during the Monetary Policy Committee (MPC) meeting held from 5 to 7 October 2026.
